# nb_editor_status

Acionado pela interface HTML `analise_siplan_editor.html` via Fabric Jobs API.  
Executa `UPDATE` em `analise_siplan_rps` para alterar o `Status` de uma atividade.

| Parâmetro | Tipo | Descrição |
|-----------|------|-----------|
| `atividade_id` | int | ID da atividade a atualizar |
| `new_status` | str | Novo valor de Status |
| `usuario` | str | E-mail do usuário que fez a alteração (preenchido pelo frontend) |

In [ ]:
# ── Parâmetros injetados pelo frontend ────────────────────────────────────────
atividade_id : int = 0      # ID da atividade a atualizar
new_status   : str = ''     # Novo Status (validado abaixo)
usuario      : str = ''     # E-mail do usuário que fez a alteração

# ── Configuração do endpoint ───────────────────────────────────────────────────
SQL_ENDPOINT  = (
    'beu5bmmdbuwedpv62ucm524jzi-j5uifqqkrzyuxjcrxjckqzjkmm'
    '.datawarehouse.fabric.microsoft.com'
)
ANALISE_TABLE = 'wh_siplan_fluxo.dbo.analise_siplan_fluxo'

In [ ]:
import json
import struct
from datetime import datetime, timedelta

try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

try:
    import pyodbc
except ImportError:
    pyodbc = None

print(f'Imports OK — {datetime.now():%d/%m/%Y %H:%M}')

In [ ]:
# ── Helpers: conexão e execução de DML ────────────────────────────────────────

def _sql_val(v) -> str:
    """Serializa valor Python para literal T-SQL seguro (sem bind params)."""
    if v is None:
        return 'NULL'
    if isinstance(v, bool):
        return '1' if v else '0'
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, datetime):
        return "'" + v.strftime('%Y-%m-%d %H:%M:%S') + "'"
    s = str(v).replace("'", "''")
    return "'" + s + "'"


def _get_jdbc_conn():
    """JDBC com token mssparkutils — uso Fabric."""
    token = mssparkutils.credentials.getToken('https://database.windows.net/')
    ds = spark._jvm.com.microsoft.sqlserver.jdbc.SQLServerDataSource()
    ds.setServerName(SQL_ENDPOINT)
    ds.setPortNumber(1433)
    ds.setEncrypt(True)
    ds.setAccessToken(token)
    return ds.getConnection()


def _get_local_conn():
    """pyodbc com token interativo — uso local."""
    from azure.identity import InteractiveBrowserCredential
    cred  = InteractiveBrowserCredential()
    token = cred.get_token('https://database.windows.net/.default').token
    tb    = token.encode('utf-16-le')
    ts    = struct.pack(f'<I{len(tb)}s', len(tb), tb)
    return pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={SQL_ENDPOINT};Encrypt=Yes;',
        attrs_before={1256: ts},
    )


def _exec_sql(sql: str) -> None:
    """DML no warehouse: JDBC no Fabric, pyodbc local."""
    if FABRIC_ENV:
        conn = _get_jdbc_conn()
        stmt = conn.createStatement()
        stmt.execute(sql)
        conn.close()
    else:
        conn = _get_local_conn()
        conn.execute(sql)
        conn.commit()
        conn.close()


print('Helpers definidos.')

In [ ]:
# ── Validação ─────────────────────────────────────────────────────────────────
STATUSES_PERMITIDOS = {
    'Enviada', 'Reenviada', 'Em revisão', 'AutonomiaUO',
    'Aprovado', 'Não aprovado',
}

assert atividade_id and int(atividade_id) > 0, \
    f'atividade_id inválido: {atividade_id!r}'
assert new_status in STATUSES_PERMITIDOS, \
    f'Status inválido: {new_status!r}. Permitidos: {STATUSES_PERMITIDOS}'

aid     = int(atividade_id)
now_brt = datetime.utcnow() + timedelta(hours=-3)

print(f'Parâmetros OK — atividade_id={aid}, new_status={new_status!r}, usuario={usuario!r}')

# ── UPDATE ────────────────────────────────────────────────────────────────────
sql = (
    f'UPDATE {ANALISE_TABLE}'
    f' SET [Status]         = {_sql_val(new_status)},'
    f'     [Modificado]     = {_sql_val(now_brt)},'
    f'     [Modificado por] = {_sql_val(usuario or "interface")}'
    f' WHERE atividade_id   = {aid}'
)

_exec_sql(sql)
print(f'UPDATE executado: atividade_id={aid} → Status={new_status!r}')

# ── Retorno ao chamador ───────────────────────────────────────────────────────
resultado = json.dumps({
    'ok':           True,
    'atividade_id': aid,
    'new_status':   new_status,
    'modificado_por': usuario,
    'modificado':   now_brt.isoformat(),
}, ensure_ascii=False)

print(resultado)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(resultado)